In [1]:
import pandas as pd
import numpy as np
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
## 데이터셋, 데이터로더 관련 모듈
from torch.nn.utils.rnn import pad_sequence
## 데이터 길이 맞추기

## 토커나이저, 단어사전 관련 모듈
from torchtext.data.utils import get_tokenizer              ## 토커나이저 인스턴스 추출
from torchtext.vocab import build_vocab_from_iterator       ## 데이터셋에서 단어사전 생성 함수
from nltk.corpus import stopwords                           ## 불용어 데이터셋
import nltk
from nltk.tokenize import word_tokenize

c:\Users\kdt\anaconda3\envs\NLP\lib\site-packages\torchtext\data\__init__.py:4: UserWarning: 
/!\ IMPORTANT WARNING ABOUT TORCHTEXT STATUS /!\ 
Torchtext is deprecated and the last released version will be 0.18 (this one). You can silence this warning by calling the following at the beginnign of your scripts: `import torchtext; torchtext.disable_torchtext_deprecation_warning()`
  warnings.warn(torchtext._TORCHTEXT_DEPRECATION_MSG)
c:\Users\kdt\anaconda3\envs\NLP\lib\site-packages\torchtext\vocab\__init__.py:4: UserWarning: 
/!\ IMPORTANT WARNING ABOUT TORCHTEXT STATUS /!\ 
Torchtext is deprecated and the last released version will be 0.18 (this one). You can silence this warning by calling the following at the beginnign of your scripts: `import torchtext; torchtext.disable_torchtext_deprecation_warning()`
  warnings.warn(torchtext._TORCHTEXT_DEPRECATION_MSG)
c:\Users\kdt\anaconda3\envs\NLP\lib\site-packages\torchtext\utils.py:4: UserWarning: 
/!\ IMPORTANT WARNING ABOUT TORCHTEXT STATU

In [2]:
dataDF = pd.read_csv('./dataDF.csv', index_col=0, encoding='utf-8')

In [3]:
texts = dataDF['Dialogue'].tolist()
labels = dataDF['age'].tolist()

In [4]:
### ===> 모듈 로딩
from konlpy.tag import Okt
from torchtext.vocab import build_vocab_from_iterator
import string
### ===> 토큰관련 특별 문자
UNK = '<UNK>'
PAD = '<PAD>'
### 토큰화 인스턴스 생성
tokenizer = Okt()
### ===> 토큰 제너레이터 함수 : 데이터 추출하여 토큰화 


In [5]:
PUNC = string.punctuation
STOPWORDS = [
    # 조사 / 불필요 접속어
    '은', '는', '이', '가', '을', '를', '에', '에서', '으로', '의', '도', '만', '까지', '부터',
    '과', '와', '하고', '보다', '보다도',

    # 보조동사 및 흔한 표현
    '있다', '없다', '되다', '해요', '해', '했어요', '했네', '했지', '하지', '그랬지', '그래요', '같아요',
    
    # 대명사 / 불분명 주어
    '그', '저', '이', '것', '거', '자기', '우리', '너', '나', '누구', '사람', '다', '뭐',

    # 웹체 / 감탄 / 의미 낮은 부사
    '정말', '그냥', '좀', '매우', '많이', '아주', '거의', '조금', '계속', '항상', '진짜',
    
    # 대화 문법 전환
    '그래서', '그런데', '그러니까', '하지만', '그리고', '그러면',

    # 의성어 / 감탄사 / 불필요한 감정어
    '티티', '아가씨', '수고했다', '감사합니다', '고맙다', '고마워서', '선물', '미신이야',

    # 웹 말투
    'ㅋㅋ', 'ㅎㅎ', 'ㅠㅠ', '...', '!!', '??'
]
UNK, PAD  = '<UNK>',  '<PAD>'

In [6]:
def yield_tokens(data):
    for line in data:
        line = ''.join([x for x in line if x not in PUNC])
        yield word_tokenize(line.lower())


In [7]:
VOCAB = build_vocab_from_iterator(yield_tokens(texts), specials=[UNK, PAD])
VOCAB.set_default_index(VOCAB[UNK])

In [8]:
VOCAB.get_itos()[:10]

['<UNK>', '<PAD>', '왜', '좀', '너', '내가', '이', '거', '그', '안']

In [9]:
# def encode_texts(data, vocab):
#     encoded = []
#     for line in data:
#         line = ''.join([c for c in line if c not in PUNC])
#         tokens = word_tokenize(line.lower())
#         token_ids = [vocab[token] for token in tokens]
#         encoded.append(torch.tensor(token_ids, dtype=torch.long))
#     return encoded

In [10]:
tokenizer = Okt()
tokens = [tokenizer.morphs(review) for review in texts]

In [11]:
from gensim.models import Word2Vec


word2vec = Word2Vec(
    sentences=tokens,
    vector_size=128,
    window=5,
    min_count=1,
    sg=1,
    epochs=10,
    max_final_vocab=10000
)

# 2. UNK 토큰 추가
unk_vector = np.random.normal(scale=0.6, size=(word2vec.vector_size,))
word2vec.wv.add_vector("<UNK>", unk_vector)

# 3. 단어 인덱스 매핑
word2index = word2vec.wv.key_to_index
unk_index = word2index["<UNK>"]

# 4. 임베딩 weight → PyTorch 임베딩 레이어
embedding_weights = torch.FloatTensor(word2vec.wv.vectors)
embedding_layer = torch.nn.Embedding.from_pretrained(embedding_weights, freeze=False)

c:\Users\kdt\anaconda3\envs\NLP\lib\site-packages\gensim\models\keyedvectors.py:551: UserWarning: Adding single vectors to a KeyedVectors which grows by one each time can be costly. Consider adding in batches or preallocating to the required size.
  warnings.warn(


In [12]:
word2vec.save("./models/word2vec.model")
word2vec = Word2Vec.load("./models/word2vec.model")

In [13]:
# encoded_sequences = encode_texts(texts, VOCAB)

# padded_sequences = pad_sequence(encoded_sequences, batch_first=True, padding_value=VOCAB["<pad>"])
# labels_tensor = torch.tensor(labels, dtype=torch.long)


In [14]:
len(tokens)

10655

In [15]:
embedding_dim = word2vec.vector_size # 로드된 모델에서 임베딩 차원 가져오기
vector_sequences = []
print("데이터를 벡터 시퀀스로 변환 중...")
for tokens in tokens:
    sequence_vectors = []
    for token in tokens:
        if token in word2vec.wv: # 모델의 어휘집(`.wv`)에 있는지 확인
            sequence_vectors.append(word2vec.wv[token]) # 단어 벡터 조회
        else:
            # OOV(Out-Of-Vocabulary) 단어 처리 (예: 0 벡터 사용)
            # print(f"Warning: OOV 단어 '{token}' 발견. 0 벡터로 대체합니다.") # OOV 로깅 (선택 사항)
            sequence_vectors.append(np.zeros(embedding_dim, dtype=np.float32))

    # 각 시퀀스를 FloatTensor로 변환하여 리스트에 추가
    if sequence_vectors:
        vector_sequences.append(torch.tensor(np.array(sequence_vectors), dtype=torch.float))
    else:
        print(f"Warning: 빈 시퀀스 발견 (토큰: {tokens}).")

print(f"변환된 시퀀스 개수: {len(vector_sequences)}")
if not vector_sequences:
    print("오류: 변환된 벡터 시퀀스가 없습니다. 입력 데이터나 Word2Vec 모델을 확인하세요.")
    exit()

데이터를 벡터 시퀀스로 변환 중...
변환된 시퀀스 개수: 10655


In [16]:
padded_sequences = pad_sequence(vector_sequences,
                                batch_first=True,
                                padding_value=0.0)
print(f"패딩된 시퀀스 텐서 모양: {padded_sequences.shape}")

# 3단계: 레이블 텐서 변환
labels_tensor = torch.tensor(labels, dtype=torch.long)
print(f"레이블 텐서 모양: {labels_tensor.shape}")

패딩된 시퀀스 텐서 모양: torch.Size([10655, 176, 128])
레이블 텐서 모양: torch.Size([10655])


In [17]:
# 4단계: 입력 텐서와 레이블 텐서의 첫 번째 차원(샘플 수) 일치 확인
if padded_sequences.shape[0] != labels_tensor.shape[0]:
    print(f"오류: 시퀀스 텐서 샘플 수({padded_sequences.shape[0]})와 "
          f"레이블 텐서 샘플 수({labels_tensor.shape[0]}) 불일치.")
    exit()
else:
    num_samples = padded_sequences.shape[0]
    print(f"총 샘플 수: {num_samples}")

총 샘플 수: 10655


In [18]:
class customDataset(Dataset):
    def __init__(self, sequences, labels):
        self.sequences = sequences
        self.labels = labels

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return self.sequences[idx], self.labels[idx]

In [19]:
from sklearn.model_selection import train_test_split

In [20]:
X_train, X_temp, y_train, y_temp = train_test_split(
    padded_sequences, labels_tensor, test_size=0.2, random_state=42, stratify=labels_tensor
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_val = le.transform(y_val)
y_test = le.transform(y_test)

train_dataset = customDataset(X_train, y_train)
val_dataset = customDataset(X_val, y_val)
test_dataset = customDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=100, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=100, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=100, shuffle=False)

In [21]:
len(train_dataset)

8524

In [22]:
train_dataset[0]
    

(tensor([[ 0.0265,  0.0446,  0.1531,  ..., -0.5208, -0.0094,  0.0264],
         [ 0.2332, -0.4094, -0.0945,  ..., -0.0261, -0.1170, -0.3886],
         [ 0.1087, -0.0284,  0.3763,  ..., -0.1509, -0.0225,  0.1040],
         ...,
         [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
         [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
         [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000]]),
 0)

In [23]:
import torch.optim as optim 
from torch.optim.lr_scheduler import StepLR

In [24]:
# ## -------------------------------------------------------------------------
# ## 클래스이름 : TextModel
# ## 부모클래스 : Module
# ## 매개변수둘 : 단어사전 갯수, 임베딩 수, 2진분류
# ## -------------------------------------------------------------------------
# class TextDnnEmbModel(nn.Module):
#     def __init__(self, vocab_size, embedding_dim=256, hidden_dim=4):
#         super().__init__()
#         self.embedding_layer = nn.Embedding(vocab_size, embedding_dim)
        
#         self.classifier = nn.Sequential(
#             nn.Linear(embedding_dim, hidden_dim),
#             nn.ReLU(),
#             nn.Dropout(0.3),
#             nn.Linear(hidden_dim, hidden_dim // 2),
#             nn.ReLU(),
#             nn.Linear(hidden_dim // 2, 4)  # 4개 클래스
#         )

#     def forward(self, text):
#         embedded = self.embedding_layer(text)  # (batch_size, seq_len, emb_dim)
#         pooled = embedded.mean(dim=1)          # (batch_size, emb_dim)
#         return self.classifier(pooled)


    

In [25]:
# import torch
# import torch.nn as nn

# class TextRNNModel(nn.Module):
#     def __init__(self, vocab_size, embedding_dim=256, hidden_dim=128, num_classes=4, num_layers=1, bidirectional=True):
#         super(TextRNNModel, self).__init__()
#         # self.embedding = nn.Embedding(vocab_size, embedding_dim)

#         self.lstm = nn.LSTM(
#             input_size=embedding_dim,
#             hidden_size=hidden_dim,
#             num_layers=num_layers,
#             batch_first=True,
#             bidirectional=bidirectional
#         )

#         direction_factor = 2 if bidirectional else 1

#         self.fc = nn.Sequential(
#             nn.Linear(hidden_dim * direction_factor, hidden_dim),
#             nn.ReLU(),
#             nn.Dropout(0.3),
#             nn.Linear(hidden_dim, num_classes)  # 4개 연대 예측
#         )

#     def forward(self, x):
#         """
#         x: (batch_size, seq_len) - 단어 인덱스로 이루어진 텐서
#         """
#         # embedded = self.embedding(x)  # (batch_size, seq_len, embedding_dim)
#         # output, (hidden, cell) = self.lstm(embedded)
#         output, (hidden, cell) = self.lstm(x)

#         # 마지막 hidden state 사용
#         if self.lstm.bidirectional:
#             hidden = torch.cat((hidden[-2], hidden[-1]), dim=1)  # (batch_size, hidden_dim*2)
#         else:
#             hidden = hidden[-1]  # (batch_size, hidden_dim)

#         logits = self.fc(hidden)  # (batch_size, num_classes)
#         return logits


In [ ]:
class TextRNNModel(nn.Module):
    def __init__(
        self,
        vocab_size,
        embedding_dim=256,
        hidden_dim=128,
        num_classes=4,
        num_layers=2,
        bidirectional=True,
        dropout=0.5
    ):
        super(TextRNNModel, self).__init__()

        # self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=bidirectional
        )

        self.bidirectional = bidirectional
        self.direction_factor = 2 if bidirectional else 1

        # Attention layer (선택)
        self.attention = nn.Linear(hidden_dim * self.direction_factor, 1)

        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden_dim * self.direction_factor),
            nn.Linear(hidden_dim * self.direction_factor, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, x):
        """
        x: (batch_size, seq_len)
        """
        # x = x.long()
        # embedded = self.embedding(x)  # (batch_size, seq_len, embedding_dim)
        # lstm_out, (hidden, cell) = self.lstm(embedded)  # lstm_out: (batch_size, seq_len, hidden*2)
        # embedded = self.embedding(x)  # (batch_size, seq_len, embedding_dim)
        lstm_out, (hidden, cell) = self.lstm(x)  # lstm_out: (batch_size, seq_len, hidden*2)

        # Attention 적용 (batch_size, seq_len, 1) → 가중합으로 문장 표현 추출
        attn_weights = torch.softmax(self.attention(lstm_out), dim=1)  # (batch, seq_len, 1)
        sentence_rep = torch.sum(attn_weights * lstm_out, dim=1)  # (batch, hidden_dim*2)

        logits = self.classifier(sentence_rep)
        return logits


In [67]:
len(word2vec.wv.key_to_index)

6683

In [76]:
## 학습 설정
INPUT_SIZE      = 256
LR              = 0.01
EPOCHS          = 10
STEP_SIZE       = 5
NUM_CLASS       = 1

EMBEDDING_DIM   = 128
HIDDEN_DIM      = 128
# VOCAB_SIZE      = len(VOCAB)
VOCAB_SIZE      = len(word2vec.wv.key_to_index)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [77]:
## 학습 인스턴스 생성
MODEL = TextRNNModel(VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_DIM)
MODEL.to(DEVICE)

LOSS_FN   = nn.CrossEntropyLoss()
OPTIMIZER = optim.Adam(MODEL.parameters(), lr=LR)
SCHEDULER = StepLR(OPTIMIZER, STEP_SIZE, gamma=0.1)
""" 
Decays the learning rate of each parameter group by gamma every step_size epochs. 
Notice that such decay can happen simultaneously with other changes to the learning rate from outside this scheduler. 
When last_epoch=-1, sets initial lr as lr.

"""
## 전에 쓴 건 스코어가 변하지 않으면 patience만큼 기다렸다가 학습중지


' \nDecays the learning rate of each parameter group by gamma every step_size epochs. \nNotice that such decay can happen simultaneously with other changes to the learning rate from outside this scheduler. \nWhen last_epoch=-1, sets initial lr as lr.\n\n'

In [78]:
for idx, (label, text) in enumerate(train_loader):
    print( idx, label.shape, text.shape)
    break

0 torch.Size([100, 176, 128]) torch.Size([100])


In [79]:
## -------------------------------------------------------------------------
## 함수기능 : 학습데이터를 사용하여 학습 진행
## 함수이름 : training
## 매개변수 : 데이터로더
## 결과반환 : 손실값, 모델성능값
## -------------------------------------------------------------------------
def training(dataloader):
    ## 학습 모드 설정
    MODEL.train()

    ## 학습 손실과 점수 저장 
    total_loss, total_acc = 0, 0
    
    for idx, (text, label) in enumerate(dataloader):

        OPTIMIZER.zero_grad()
        pre  = MODEL(text)

        loss = LOSS_FN(pre, label.reshape(-1).long())
        loss.backward()
        ## gradient vanishing, gradient exploding 발생 => 방지 및 안정화 
        ## - gradient가 일정 threshold를 넘어가면 clipping
        ## - clipping: gradient의 L2norm(norm이지만 보통 L2 norm사용)으로 나눠주는 방식
        torch.nn.utils.clip_grad_norm_(MODEL.parameters(), 0.1)
        OPTIMIZER.step()

        total_loss += loss.item()
        total_acc += (pre.argmax(dim=1) == label).sum().item()

        if idx==5: break
        
    return total_loss/idx+1, total_acc/idx+1



In [80]:
## -------------------------------------------------------------------------
## 함수기능 : 검증데이터를 사용하여 학습 진행
## 함수이름 : evaluate
## 매개변수 : 데이터로더
## 결과반환 : 손실값, 모델성능값
## -------------------------------------------------------------------------
def evaluate(dataloader):
    MODEL.eval()
    total_loss, total_acc = 0, 0

    with torch.no_grad():
        for idx, (text, label) in enumerate(dataloader):
            ## 추론 진행
            pre = MODEL(text)
            ## 손실 계산
            loss = LOSS_FN(pre, label.reshape(-1).long())

            ## 손실 및 성능 평가
            total_loss += loss.item()
            total_acc += (pre.argmax(1) == label).sum().item()
            if idx==5: break
        
    return total_loss/idx+1, total_acc/idx+1

In [81]:
## 모델 및 모델 층별 상태값 즉, 파라미터 값 저장 경로
MODEL_DIR  = './models/'
MODEL_FILE = 'IMDB_DNN_MODEL.pt'


In [82]:
# EPOCHS = 100  ## 임시
# # 모델 저장 기준
# MAX_ACC = 0.

# for epoch in range(1, EPOCHS + 1):
    
#     train_loss, train_acc = training(train_loader)
#     valid_loss, valid_acc = evaluate(val_loader)
#     SCHEDULER.step()

#     print("-" * 59)
#     print(f'| end of epoch {epoch:3d} | train acc {train_acc:8.3f}  | valid acc {valid_acc:8.3f}')
#     print("-" * 59)

#     ## 모델 저장 
#     if MAX_ACC < valid_acc : 
#         torch.save(MODEL, MODEL_DIR+MODEL_FILE)
#         MAX_ACC = valid_acc


In [83]:
import os

EPOCHS = 100
PATIENCE = 25
patience_counter = 0
MAX_ACC = 0.

for epoch in range(1, EPOCHS + 1):
    
    train_loss, train_acc = training(train_loader)
    valid_loss, valid_acc = evaluate(val_loader)
    SCHEDULER.step()

    print("-" * 59)
    print(f'| end of epoch {epoch:3d} | train acc {train_acc:8.3f}  | valid acc {valid_acc:8.3f}')
    print("-" * 59)

    # 모델 저장 디렉토리 없으면 생성
    os.makedirs(MODEL_DIR, exist_ok=True)

    # 성능 향상 시 저장 (epoch 번호 포함)
    if valid_acc > MAX_ACC and valid_acc > 60:
        model_path = os.path.join(MODEL_DIR, f"epoch{epoch}_v{valid_acc:.2f}.pt")
        torch.save(MODEL, model_path)
        MAX_ACC = valid_acc
        patience_counter = 0
        print(f"✔ 모델 저장됨: {model_path}")
    else:
        patience_counter += 1
        print(f'→ No improvement. Patience counter: {patience_counter}/{PATIENCE}')

        if patience_counter >= PATIENCE:
            print(f'→ Early stopping at epoch {epoch} (best val acc: {MAX_ACC:.3f})')
            break


-----------------------------------------------------------
| end of epoch   1 | train acc   27.400  | valid acc   34.400
-----------------------------------------------------------
→ No improvement. Patience counter: 1/25
-----------------------------------------------------------
| end of epoch   2 | train acc   29.000  | valid acc   29.600
-----------------------------------------------------------
→ No improvement. Patience counter: 2/25
-----------------------------------------------------------
| end of epoch   3 | train acc   30.800  | valid acc   29.600
-----------------------------------------------------------
→ No improvement. Patience counter: 3/25
-----------------------------------------------------------
| end of epoch   4 | train acc   31.800  | valid acc   34.400
-----------------------------------------------------------
→ No improvement. Patience counter: 4/25
-----------------------------------------------------------
| end of epoch   5 | train acc   37.000  | valid